# Ch01-02: 결측값 메커니즘과 다중 대치법 — 모범답안


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 12
np.random.seed(42)

---
## 문제 1 풀이

MCAR/MAR/MNAR 결측 생성 및 편향 분석

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns
np.random.seed(42)
n = 1000
mean = [50,30,20,40,60]
cov_m = np.array([[100,30,-10,20,15],[30,64,5,-10,8],[-10,5,25,12,-5],[20,-10,12,49,10],[15,8,-5,10,81]])
data = np.random.multivariate_normal(mean, cov_m, n)
cols = ['V1','V2','V3','V4','V5']
df_full = pd.DataFrame(data, columns=cols)

df_mcar = df_full.copy()
for c in cols: df_mcar.loc[np.random.random(n)<0.2, c] = np.nan

df_mar = df_full.copy()
df_mar.loc[np.random.random(n)<1/(1+np.exp(-(data[:,0]-55)/8)), 'V2'] = np.nan

df_mnar = df_full.copy()
df_mnar.loc[np.random.random(n)<1/(1+np.exp(-(data[:,1]-35)/5)), 'V2'] = np.nan

# 상관행렬 비교
fig, axes = plt.subplots(1,4,figsize=(20,4))
for ax, t, d in zip(axes, ['원본','MCAR','MAR','MNAR'], [df_full,df_mcar,df_mar,df_mnar]):
    sns.heatmap(d.corr(), annot=True, fmt='.2f', ax=ax, vmin=-1, vmax=1, cmap='RdBu_r'); ax.set_title(t)
plt.tight_layout(); plt.show()

print("=== 평균 편향 ===")
for name, d in [('MCAR',df_mcar),('MAR',df_mar),('MNAR',df_mnar)]:
    bias = d.mean()-df_full.mean()
    print(f"{name}: {dict(zip(cols, bias.round(3)))}")


**결과 해석**: MCAR은 편향 무시 가능. MAR은 조건 변수에 의존하여 일부 변수 편향. MNAR은 결측 변수 자체의 분포가 왜곡되어 가장 심각한 편향 발생.

---
## 문제 2 풀이

Little's MCAR 검정 EM 기반 구현

In [ ]:
import numpy as np, pandas as pd
from scipy import stats

def littles_mcar_test(df):
    df = df.copy(); p = df.shape[1]; n = df.shape[0]
    R = (~df.isnull()).astype(int)
    patterns = R.drop_duplicates()
    pattern_keys = [tuple(row) for _, row in patterns.iterrows()]

    complete = df.dropna()
    mu = complete.mean().values if len(complete)>5 else df.mean().values
    sigma = (complete.cov().values if len(complete)>5 else df.cov().values)
    sigma = np.nan_to_num(sigma) + np.eye(p)*0.01

    for _ in range(50):
        mu_old = mu.copy()
        T1, T2 = np.zeros(p), np.zeros((p,p))
        for i in range(n):
            row = df.values[i]
            obs = np.where(~np.isnan(row))[0]; mis = np.where(np.isnan(row))[0]
            if len(mis)==0:
                T1 += row; T2 += np.outer(row,row); continue
            if len(obs)==0:
                T1 += mu; T2 += np.outer(mu,mu)+sigma; continue
            sig_oo_inv = np.linalg.pinv(sigma[np.ix_(obs,obs)])
            cm = mu[mis] + sigma[np.ix_(mis,obs)]@sig_oo_inv@(row[obs]-mu[obs])
            cc = sigma[np.ix_(mis,mis)] - sigma[np.ix_(mis,obs)]@sig_oo_inv@sigma[np.ix_(obs,mis)]
            xf = row.copy(); xf[mis] = cm; T1 += xf
            outer = np.outer(xf,xf)
            for a,ma in enumerate(mis):
                for b,mb in enumerate(mis): outer[ma,mb] += cc[a,b]
            T2 += outer
        mu = T1/n; sigma = T2/n - np.outer(mu,mu)
        if np.max(np.abs(mu-mu_old))<1e-6: break

    d2, df_test = 0, 0
    for pat in pattern_keys:
        obs_v = np.where(np.array(pat)==1)[0]
        if len(obs_v)==0: continue
        mask = np.all(R.values==np.array(pat), axis=1); nj = mask.sum()
        if nj<2: continue
        mean_j = np.nanmean(df.values[mask][:,obs_v], axis=0)
        diff = mean_j - mu[obs_v]
        d2 += nj * diff @ np.linalg.pinv(sigma[np.ix_(obs_v,obs_v)]) @ diff
        df_test += len(obs_v)
    df_test = max(df_test - p, 1)
    return {'statistic': d2, 'df': df_test, 'p_value': 1-stats.chi2.cdf(d2, df_test)}

np.random.seed(42)
data = np.random.multivariate_normal([50,30,20], [[100,30,-10],[30,64,5],[-10,5,25]], 500)
df_f = pd.DataFrame(data, columns=['A','B','C'])
df_mc = df_f.copy()
for c in ['A','B','C']: df_mc.loc[np.random.random(500)<0.2, c] = np.nan
df_ma = df_f.copy()
df_ma.loc[np.random.random(500)<1/(1+np.exp(-(data[:,0]-55)/5)), 'B'] = np.nan

r1 = littles_mcar_test(df_mc); r2 = littles_mcar_test(df_ma)
print(f"MCAR data: stat={r1['statistic']:.3f}, p={r1['p_value']:.4f}")
print(f"MAR data:  stat={r2['statistic']:.3f}, p={r2['p_value']:.4f}")


**결과 해석**: MCAR 데이터에서는 p-value가 높아 MCAR 기각 못 함. MAR 데이터에서는 패턴별 평균 차이가 유의하여 MCAR 기각. Little's test는 MAR/MNAR 구분은 불가.

---
## 문제 3 풀이

Bayesian MICE 직접 구현 + Rubin's rule

In [ ]:
import numpy as np, pandas as pd
np.random.seed(42)
n=500; X1=np.random.normal(0,1,n); X2=np.random.normal(0,1,n)
Y = 3+2*X1-1.5*X2+np.random.normal(0,1,n)
df_full = pd.DataFrame({'X1':X1,'X2':X2,'Y':Y})
df_miss = df_full.copy()
df_miss.loc[np.random.random(n)<1/(1+np.exp(-(X1-0.5)))*0.4, 'Y'] = np.nan

def mice_impute(df, m=5, max_iter=20):
    cols = df.columns.tolist(); imputed = []
    for imp in range(m):
        rng = np.random.RandomState(42+imp); di = df.copy()
        for c in cols:
            mask = di[c].isnull()
            if mask.any(): di.loc[mask,c] = di[c].mean()
        for _ in range(max_iter):
            for tc in cols:
                miss = df[tc].isnull()
                if not miss.any(): continue
                preds = [c for c in cols if c!=tc]; obs = ~df[tc].isnull()
                Xo = di.loc[obs, preds].values; yo = df.loc[obs, tc].values
                Xd = np.column_stack([np.ones(obs.sum()), Xo])
                XtXi = np.linalg.inv(Xd.T@Xd+np.eye(Xd.shape[1])*1e-6)
                bh = XtXi@Xd.T@yo; res = yo-Xd@bh; s2 = np.sum(res**2)/max(obs.sum()-Xd.shape[1],1)
                dof = max(obs.sum()-Xd.shape[1],1)
                s2d = dof*s2/rng.chisquare(dof)
                bd = rng.multivariate_normal(bh, s2d*XtXi)
                Xm = np.column_stack([np.ones(miss.sum()), di.loc[miss,preds].values])
                di.loc[miss,tc] = Xm@bd + rng.normal(0,np.sqrt(s2d),miss.sum())
        imputed.append(di.copy())
    return imputed

def rubins_rule(est, var):
    m=len(est); Qb=np.mean(est); Ub=np.mean(var); B=np.var(est,ddof=1)
    T=Ub+(1+1/m)*B; return {'estimate':Qb, 'se':np.sqrt(T), 'within':Ub, 'between':B}

dsets = mice_impute(df_miss, m=5)
b1s, v1s = [], []
for di in dsets:
    Xm = np.column_stack([np.ones(n), di['X1'].values, di['X2'].values])
    XtXi = np.linalg.inv(Xm.T@Xm); b=XtXi@Xm.T@di['Y'].values
    s2=np.sum((di['Y'].values-Xm@b)**2)/(n-3)
    b1s.append(b[1]); v1s.append(s2*XtXi[1,1])
r = rubins_rule(b1s, v1s)
print(f"결합 beta1: {r['estimate']:.4f} (참값: 2.0)")
print(f"95% CI: [{r['estimate']-1.96*r['se']:.4f}, {r['estimate']+1.96*r['se']:.4f}]")


**결과 해석**: Bayesian MICE는 각 대치에 불확실성을 반영한다. Rubin's rule은 within/between variance를 결합하여 올바른 추론을 보장한다.

---
## 문제 4 풀이

MNAR 민감도 분석: 틸팅 매개변수에 따른 편향

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer
np.random.seed(42)
n=1000; X1=np.random.normal(0,1,n); X2=np.random.normal(0,1,n)
Y = 3+2.0*X1-1.5*X2+np.random.normal(0,1,n)
Xm = np.column_stack([np.ones(n),X1,X2])
bt = np.linalg.lstsq(Xm, Y, rcond=None)[0]

alphas = [0,0.3,0.6,1.0,1.5,2.0]
res = {'a':[],'cc':[],'mice':[],'bias_cc':[],'bias_mice':[]}
for a1 in alphas:
    prob = 1/(1+np.exp(-(-1+a1*Y)))
    df_m = pd.DataFrame({'X1':X1,'X2':X2,'Y':Y})
    df_m.loc[np.random.random(n)<prob,'Y'] = np.nan
    cc = df_m.dropna(); Xc = np.column_stack([np.ones(len(cc)),cc['X1'],cc['X2']])
    bc = np.linalg.lstsq(Xc, cc['Y'].values, rcond=None)[0]
    di = pd.DataFrame(IterativeImputer(max_iter=20,random_state=42).fit_transform(df_m), columns=df_m.columns)
    Xi = np.column_stack([np.ones(n),di['X1'],di['X2']])
    bi = np.linalg.lstsq(Xi, di['Y'].values, rcond=None)[0]
    res['a'].append(a1); res['cc'].append(bc[1]); res['mice'].append(bi[1])
    res['bias_cc'].append(bc[1]-bt[1]); res['bias_mice'].append(bi[1]-bt[1])

fig, axes = plt.subplots(1,2,figsize=(14,5))
axes[0].plot(res['a'],res['cc'],'bo-',lw=2,label='CC')
axes[0].plot(res['a'],res['mice'],'rs-',lw=2,label='MICE')
axes[0].axhline(bt[1],color='green',ls='--',lw=2,label=f'참값({bt[1]:.2f})')
axes[0].set_xlabel('α₁'); axes[0].set_ylabel('β̂₁'); axes[0].legend(); axes[0].set_title('추정치')
axes[1].plot(res['a'],res['bias_cc'],'bo-',lw=2,label='CC')
axes[1].plot(res['a'],res['bias_mice'],'rs-',lw=2,label='MICE')
axes[1].axhline(0,color='green',ls='--'); axes[1].set_xlabel('α₁'); axes[1].set_ylabel('편향')
axes[1].set_title('편향'); axes[1].legend()
plt.tight_layout(); plt.show()


**결과 해석**: α₁=0(MCAR)에서는 편향 없음. α₁ 증가(MNAR 강도)에 따라 CC와 MICE 모두 편향 증가. MNAR에서는 결측 메커니즘 민감도 분석이 필수적이다.

---
## 확장 토론

### 결측 처리 전략

1. 결측률 < 5%: CC 수용 가능
2. 5-30%, MCAR/MAR: MICE 권장
3. > 30%: 민감도 분석 필수
4. MNAR 의심: Selection model 또는 Pattern-Mixture model